# Security level validation

This notebook compares the structural and regression scenario models for one company. Choose the ticker below and run the notebook from top to bottom. It reads the existing simulation exports, explains each validation, and saves tables that can be used directly in an HTML dashboard.

The initial company is **NOG**. Financial and numerical checks can pass or fail. Historical and observed-outcome comparisons are descriptive; an unavailable test is reported as **unassessed**, with its reason. Existing hypothetical scenarios are not relabeled as out-of-sample forecasts.

The first code cell also contains the required Python packages. To install them into the active Python environment, set `INSTALL_DEPENDENCIES = True` before running that cell. Leave it `False` when the packages are already installed. If you install packages in a running notebook, restart the kernel before running the validation.

In [ ]:
# Change COMPANY to analyze another exported company.
from pathlib import Path
import os
import sys

COMPANY = "NOG"
YEARS = [2024, 2025, 2026]
PROJECT_ROOT = Path("/Users/gezimzeneli/Documents/ChatGPT/MScThesisV2")
AS_OF = "2026-11-08"  # Explicit observation cutoff, recorded in the results.
RESULTS_ROOT = Path(os.environ.get("SECURITY_VALIDATION_OUTPUT_ROOT", str(PROJECT_ROOT / "validation" / "security_level")))
CONTROL_EXPORT_ROOT = Path(os.environ.get("SECURITY_VALIDATION_CONTROL_ROOT", str(PROJECT_ROOT / "export")))

# Explicit V2 run: change both COMPANY and this directory for another security.
# Set None to skip the independent V2 section. No automatic latest-run discovery.
V2_RUN_DIRECTORY = PROJECT_ROOT / "export/v2/20261111T235336081947Z/garch_events/NOG/structured"
V2_RESULTS_ROOT = Path(os.environ.get("SECURITY_VALIDATION_V2_OUTPUT_ROOT", str(PROJECT_ROOT / "validation/security_level_v2")))


HISTORICAL_LOOKBACK_YEARS = 8
BOOTSTRAP_REPLICATIONS = 1000
BOOTSTRAP_BLOCK_LENGTH = 20
BOOTSTRAP_SENSITIVITY_LENGTHS = [10, 40]
ZERO_REFERENCE_POLICY = "supplied_zero_shock"  # User-defined 0% event reference.
EXPERIMENT_SCOPE = "exported_relative_sweeps"
SHOCK_MAGNITUDES = [-1.0, -0.5, -0.3, -0.2, -0.1, 0.1, 0.2, 0.3, 0.5, 1.0]  # Relative input changes; 0% uses zero_shock.
RESPONSE_EXPECTATIONS = []  # Optional preregistered directions; see README.
ENERGY_SCORE_PAIRS = 200000
SAVE_INDIVIDUAL_PATH_METRICS = False  # Summary tables are much smaller for dashboards.
RANDOM_SEED = 20261108

# Dependencies for this notebook. Enable installation when setting up a new environment.
INSTALL_DEPENDENCIES = False
REQUIRED_PACKAGES = [
    "numpy>=1.26",
    "pandas>=2.1",
    "pyarrow>=14",
    "pandas-market-calendars>=4",
    "nbformat>=5.9",
    "nbclient>=0.9",
    "ipykernel>=6",
    "scipy>=1.11",       # Historical ARCH-LM chi-square reference.
    "matplotlib>=3.8",   # Exported V2 scientific figures.
]
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES], check=True)

module_root = next((p for p in [Path.cwd(), PROJECT_ROOT] if (p / "security_validation").is_dir()), None)
if module_root is None:
    raise FileNotFoundError("Keep the security_validation folder next to this notebook, or set PROJECT_ROOT correctly.")
sys.path.insert(0, str(module_root))
from security_validation.engine import ValidationSession, default_config
import pandas as pd
from IPython.display import display

config = default_config(PROJECT_ROOT, COMPANY, RESULTS_ROOT)
config.update(years=YEARS, as_of=AS_OF,
              historical_lookback_years=HISTORICAL_LOOKBACK_YEARS,
              bootstrap_reps=BOOTSTRAP_REPLICATIONS, bootstrap_block_length=BOOTSTRAP_BLOCK_LENGTH,
              bootstrap_sensitivity_lengths=BOOTSTRAP_SENSITIVITY_LENGTHS, response_expectations=RESPONSE_EXPECTATIONS,
              energy_pairs=ENERGY_SCORE_PAIRS, write_path_metrics=SAVE_INDIVIDUAL_PATH_METRICS,
              seed=RANDOM_SEED, control_export_root=str(CONTROL_EXPORT_ROOT),
              zero_reference_policy=ZERO_REFERENCE_POLICY, experiment_scope=EXPERIMENT_SCOPE,
              shock_magnitudes=SHOCK_MAGNITUDES)
session = ValidationSession(config)
print(f"Company: {COMPANY}; years: {YEARS}; planned scenario runs: {len(session.catalog)}")
print(f"Latest available observed close: {session.observed.index.max().date()}")
print(f"Result folder: {session.output_dir}")

def preview(test_id, table, columns=None, limit=12):
    display(session.test_results(test_id))
    frame = session.table(table)
    if not frame.empty:
        if "test_id" in frame:
            frame = frame[frame.test_id.astype(str).str.startswith(test_id)]
        if columns:
            frame = frame[[c for c in columns if c in frame]]
        display(frame.head(limit))
    else:
        print("No rows in this table; see the issue and required-experiment tables.")


## T00 Input integrity and provenance

Before comparing the models, we check that the files can be read and that their prices, dates, row counts and recorded source versions agree. A broken input prevents calculations that depend on it. Changes to today’s generator do not invalidate a verified older export.

The requested grid includes 0%, ±10%, ±20%, ±30%, ±50% and ±100%. We read the original and shock-expansion manifests and verify each run against its own recorded sources. A documented unsupported input is distinguished from a missing or corrupted export. In the supplied expansion, regression oil −100% cannot be generated because its logarithmic input is undefined; it receives no assumed return or score. The inventory shows every requested case, including these limitations.

In [ ]:
session.prepare()
preview("T00", "integrity_checks")
inventory = session.table("run_inventory")
display(inventory.groupby(["year", "model", "availability"], dropna=False).size().rename("runs").reset_index())
unavailable = inventory.loc[~inventory.eligible_for_price_diagnostics]
if not unavailable.empty:
    display(unavailable[["year", "model", "scenario", "magnitude", "availability", "reason", "limitation"]])
print("Supported price diagnostics:", int(inventory.eligible_for_price_diagnostics.sum()), "of", len(inventory), "planned runs")
print("Model comparisons use only matched eligible cases; inspect scenario_coverage in the exported tables.")


## T01 Accounting and valuation

We rebuild financial relationships from the underlying accounts and inspect earnings per share, valuation and cash-default rules. These checks use each run’s recorded inputs. Company-specific identities that are not supported by an independent mapping remain unassessed rather than being guessed.

In [ ]:
session.accounting()
preview("T01", "accounting_checks")

## T02 Supplied zero reference and additional control diagnostics

The experiment defines `regression_zero_shock` and `structured_zero_shock` as the intended zero-magnitude reference. We use these supplied exports as 0% for their corresponding model and year and record that definition in the results. This is not counted as an independent numerical pass obtained by comparing a file with itself.

When separately generated, paired event controls are available, their comparisons remain visible as additional implementation diagnostics. They do not replace the supplied 0% reference used in T03.

The current experiment scope uses the supplied isolated relative-shock sweeps. Mixed-channel neutralization checks are outside this scope and are not counted as missing required tests.


In [ ]:
session.experiments()
preview("T02", "zero_reference_definitions")
display(session.table("zero_shock_controls"))


## T03 Shock magnitude and timing

We compare −100%, −50%, −30%, −20%, −10%, 0%, +10%, +20%, +30%, +50% and +100% input shocks. The supplied `zero_shock` folder is the shared event with 0% magnitude for each model and year. A shock percentage describes the input factor, not the resulting equity return. Interest-rate shocks remain relative changes, not basis-point changes.

We measure the response relative to 0%, the extra change between neighboring magnitudes, and positive/negative asymmetry. The gap from 30% to 50% is twice the gap from 20% to 30%; the gap from 50% to 100% is larger again. We therefore report slopes and curvature normalized by the actual shock spacing. Raw equal-step second differences are retained only where adjacent gaps are equal.

The −100% oil response is unavailable for regression in the supplied exports. We retain the valid structural response and regression +100% response, but cannot calculate a matched two-model comparison or regression ±100% asymmetry. Missing endpoints remain visible rather than being filled with zero or a smaller substitute shock.

Timing results show when effects appear and persist. Additional event-control outcomes stay separate from the supplied zero-reference definition. Absolute basis-point experiments and mixed-channel neutralization remain outside this experiment scope. Observed ordering is descriptive; an economic pass/fail verdict requires a predefined response expectation.


In [ ]:
preview("T03", "shock_sensitivity")
print("Coverage of every requested shock, including the 0% reference:")
display(session.table("shock_grid_coverage")[["year", "model", "shock_family", "shock_magnitude", "export_status", "reason"]])
# Show the newly added cases explicitly so they are not hidden by head().
summary = session.table("run_summaries")
if not summary.empty and {"magnitude", "metric"}.issubset(summary):
    expanded = summary.loc[pd.to_numeric(summary["magnitude"], errors="coerce").abs().isin([0.5, 1.0]) & summary.metric.isin(["mean_return", "loss_probability", "default_probability"])]
    display(expanded[["year", "model", "scenario", "magnitude", "metric", "value", "status"]])
else:
    print("No eligible scenario outcomes are available; the coverage and issue tables explain why.")
for table_name in ["shock_increments", "shock_shape", "shock_timing"]:
    print(table_name)
    display(session.table(table_name).head(12))
    records = session.table(table_name)
    if "status" in records:
        gaps = records.loc[records.status.astype(str).str.startswith("unassessed")]
        if not gaps.empty:
            display(gaps.head(12))

# Explicitly inspect unequal 30% → 50% → 100% intervals and their normalized slopes.
increments = session.table("shock_increments")
if not increments.empty:
    expanded_steps = increments.loc[pd.to_numeric(increments["to_magnitude"], errors="coerce").abs().isin([0.5, 1.0])]
    display(expanded_steps[["year", "model", "shock_family", "metric", "comparison", "from_magnitude", "to_magnitude", "value", "unit", "slope", "slope_unit", "status", "reason"]].head(40))


## T04 Return moments and tails

We compare average returns, volatility, skewness, kurtosis and extreme-return quantiles with historical periods of the same length. Both models use the same historical reference. These diagnostics require complete positive-price paths, and the excluded paths are counted. Historical uncertainty is different from the spread of simulated scenarios.

Some extreme structural scenarios default every simulated path. They still contain valid total-loss outcomes, but have no complete positive-price paths for daily log-return moments and dependence tests. Those statistics remain undefined. The same applies when a low- or high-volatility subgroup is empty: this is a cohort-coverage finding, not many separate calculation failures. Overall outcomes, including defaults, remain in T09–T11 and T13.


In [ ]:
session.return_diagnostics()
preview("T04", "historical_summary")
eligibility = session.table("eligibility")
if not eligibility.empty:
    display(eligibility.loc[eligibility.metric.eq("complete_positive_paths"),
            ["year", "model", "scenario", "magnitude", "value", "n_assessed", "n_excluded"]])
regimes = session.table("historical_summary")
if "volatility_regime" in regimes:
    regimes = regimes.loc[regimes.metric.eq("sample_sd") & regimes.volatility_regime.notna()]
    display(regimes[["year", "model", "scenario", "magnitude", "volatility_regime", "value", "n_assessed", "status"]])


## T05 Wasserstein distance

This measures the overall gap between the simulated and historical daily return distributions. Smaller distances mean closer resemblance to the chosen history. Valid returns before a later default are included, while transitions to zero have no finite log return and are counted separately.

In [ ]:
preview("T05", "historical_summary")

## T06 Autocorrelation and leverage

We inspect whether returns and large movements tend to follow one another, and whether losses are followed by larger fluctuations. Correlations are calculated within each path at lags of 1 to 20 trading days. The uncertainty bands are pointwise historical bands, not a certificate that the whole profile is correct.

In [ ]:
preview("T06", "lag_profiles")

## T07 Ljung–Box serial correlation

This checks whether serial correlations jointly differ from zero for raw, absolute and squared returns. We use 20 lags and repeat the calculation with 10. A low p-value indicates dependence in that series; it does not mean that the simulation model has failed.

In [ ]:
preview("T07", "ljung_box")

## T08 Multi-day volatility

We divide returns into blocks of 5, 10 and 20 trading days and compare the size, variation and persistence of price movements. Incomplete blocks are counted. A year contains few long blocks, so persistence estimates should be interpreted carefully.

In [ ]:
preview("T08", "volatility_blocks")

## T09 CRPS and mean bias

We compare the complete simulated return distribution with the realized return on exactly matching dates. Lower CRPS means better agreement for that particular case; bias shows whether the simulated mean is too high or low. Zero-price losses remain in the distribution. Hypothetical-scenario scores are descriptive, and missing future observations are left unscored.

In [ ]:
session.observed_diagnostics()
preview("T09", "observed_scores")
display(session.table("model_comparisons").head(12))

## T10 Interval coverage and precision

We construct central 80%, 90% and 95% ranges and record whether the observed return falls inside them. Interval width measures precision, and the interval score penalizes both wide ranges and missed observations. One hit or miss does not establish calibrated coverage.

In [ ]:
preview("T10", "interval_scores")

## T11 Maximum drawdown

We calculate the largest decline from an earlier price peak, including the initial price. This reveals losses that may be hidden by a recovery before year-end. Simulated and observed drawdowns use the same completed dates; reaching zero produces a 100% drawdown.

In [ ]:
preview("T11", "drawdowns")

## T12A Conditional historical factor replay — separate design

A conditional replay supplies the factors that actually occurred during the held-out year. It asks whether the model explains the stock path given those realized factors. This differs from a forecast made at the start of the year.

The available baseline holdouts in T12B use no inserted future shock events. Their results are assessed even when no conditional-replay export exists. The replay registry remains visible so these two designs are not confused.


In [ ]:
session.out_of_sample()
preview("T12", "oos_case_registry")

## T12B Out-of-sample baseline forecasts for the selected company

Here we compare the model distributions with what actually happened in the following year. Both models use the same January forecast origin, the same trading dates, and the same zero-event baseline. Model fits stop before the origin. Structural forecasts use original annual filings and share counts available at that time, dated credit-rating snapshots, and a pre-origin default-rate study.

The historical inputs and predictions are reconstructed separately for each company. This notebook loads those saved results, verifies their file hashes and dates, checks that both models cover the same period, and exports the scores for the HTML dashboard. It never substitutes NOG's results for another company. All defaulted paths remain in outcome scores; any genuinely unidentified prices remain explicitly recorded.

The study is a **retrospective temporal holdout**: the prediction inputs precede the evaluated year, but the model specification was selected retrospectively. These are baseline forecasts, not a replay of realized future factor shocks. Each company has only two completed annual cases (2024 and 2025); 10,000 simulated paths do not create 10,000 observed years. Read the company-specific source chronology and limitations with its scores.


In [ ]:
# The same loader is called by session.run_all(), so script and notebook results agree.
from security_validation.temporal_holdout import archive_root

TEMPORAL_HOLDOUT_ROOT = archive_root(session.config)
holdout_summary = session.temporal_holdout()
print(f"Company-specific historical baseline archive: {TEMPORAL_HOLDOUT_ROOT}")
if holdout_summary.empty:
    display(session.table("temporal_holdout_availability"))
else:
    print(f"Loaded {COMPANY} baseline holdouts for years {sorted(holdout_summary.year.unique())}.")
    print("Both model outputs use matching completed annual horizons. Default losses remain included.")
    print("Conditional factor replay above is a separate design. These saved forecasts are not recomputed by loading them.")
    display(holdout_summary)
    display(session.table("temporal_holdout_model_comparison"))
    display(session.table("temporal_holdout_case_registry"))


## T13 Joint return paths

We score a security’s returns jointly at quarter-end dates using energy and variogram scores. This checks aspects of the path across time that a terminal-return score can miss. Full-year and partial-year results are labeled separately, and zero-price outcomes are retained. These are comparisons within one company, not portfolio tests.

In [ ]:
preview("T13", "joint_scores")

## Export results for the dashboard

The export contains a manifest and one JSON, Parquet and CSV file per result table. JSON is suitable for browser-based dashboards, Parquet preserves efficient analytical data, and CSV is convenient for inspection. The manifest describes the tables, units, configuration, source hashes and model versions. A new notebook session creates a new result folder.

Always display status and exclusion counts alongside numerical results. Missing experiments and historical inputs remain visible in the issue, control and out-of-sample tables.

`run_inventory` and `scenario_coverage` distinguish successful exports from documented model limitations and show where both models can be compared. `test_status_by_year` provides counts for a selected year; `test_status_global` contains checks without a year. The all-years `test_summary` is separate. Result-table hashes and original or expansion manifest paths are retained for reproducibility. New shock sweeps do not create missing historical-input out-of-sample replays.


In [ ]:
result_folder = session.export()
import json
manifest = json.loads((result_folder / "dashboard_manifest.json").read_text())
display(pd.DataFrame([{"table": d["name"], "rows": d["rows"], "JSON": d["files"]["json"], "Parquet": d["files"]["parquet"]} for d in manifest["datasets"]]))
print(f"Dashboard entry point: {result_folder / 'dashboard_manifest.json'}")
issues = session.table("issues")
if not issues.empty:
    print("Review these unavailable or affected calculations:")
    display(issues)

print("Test-status counts by year (use these for the dashboard year selector):")
display(session.table("test_status_by_year"))
if not session.table("test_status_global").empty:
    print("Shared checks without a year:")
    display(session.table("test_status_global"))


## V2 GARCH validation — separate selected run

This section checks the new volatility process for the explicitly selected `V2_RUN_DIRECTORY` above. It preserves the earlier T00–T13 workflow and its exports. The company, model version, volatility mode and source run ID stay attached to every new result. Change the directory and company together; a mismatch stops analysis. Set the directory to `None` to skip V2.

Volatility describes the size of random daily movements. GARCH lets that size change: a large random movement can raise the following day's variance. The additional event variance is a scenario assumption. A prescribed price jump is kept separate from the random innovation used by GARCH.

We first verify every declared table hash, the NYSE trading dates, path IDs, default and zero-target alignment, and decimal-return units. A corrupted or misaligned input stops dependent analysis. Hash agreement establishes consistency with the run metadata, not independent proof of the source data's truth.

In [ ]:
from security_validation.v2 import V2ValidationSession
from security_validation.v2_plots import dependence_figure, historical_residual_figure, event_figures
from security_validation.v2_report import findings, write_report

v2_session = None
if V2_RUN_DIRECTORY is not None:
    v2_session = V2ValidationSession(V2_RUN_DIRECTORY, V2_RESULTS_ROOT,
                                    expected_ticker=COMPANY,
                                    bootstrap_reps=BOOTSTRAP_REPLICATIONS, seed=RANDOM_SEED)
    v2_session.prepare()
    display(pd.DataFrame([v2_session.run.identity]))
    display(v2_session.table("v2_integrity"))
else:
    print("V2 section disabled: V2_RUN_DIRECTORY is None.")

def v2_show(name, columns=None):
    frame = v2_session.table(name)
    if frame.empty:
        print(f"{name}: no assessed rows; see replay/integrity checks for the reason.")
    else:
        display(frame if columns is None else frame[[c for c in columns if c in frame]])


def v2_show_figure(figure):
    from io import StringIO
    from IPython.display import SVG
    import matplotlib.pyplot as plt
    buffer = StringIO()
    figure.savefig(buffer, format="svg", bbox_inches="tight")
    display(SVG(buffer.getvalue()))
    plt.close(figure)  # The retained figure can still be exported; avoid duplicate automatic displays.


### V2 / T06 — Are large movements followed by other large movements?

We reuse the existing raw, absolute and squared-return autocorrelations at lags 1–20, plus the leverage correlation. Every correlation is calculated within one path. Independent paths are never concatenated.

Two views are shown: **total daily log returns**, which include prescribed jumps, and **returns with those dated log jumps subtracted**. The second view still includes drift and the GARCH variance correction; it is not the standardized innovation series. This separation prevents one large prescribed jump from being confused with persistent random volatility.

The historical reference uses the latest nonoverlapping calibration windows with exactly the simulation's length. Bands describe pointwise bootstrap uncertainty conditional on those windows. Defaulting paths, zero-equity targets and incomplete/nonpositive price paths are excluded from these full-path log-return diagnostics, with counts shown below. These exclusions apply only to full-path dependence diagnostics. This V2 section does not compute new outcome scores; the existing legacy outcome results remain unchanged. No interpolation or gap bridging is used.

In [ ]:
if v2_session is not None:
    v2_session.dependence()
    display(v2_session.table("v2_return_eligibility"))
    display(v2_session.table("v2_historical_reference_eligibility"))
    display(v2_session.table("v2_historical_windows"))
    fig = dependence_figure(v2_session)
    v2_session.figures["v2_return_dependence"] = fig
    v2_show_figure(fig)
    frame = v2_session.table("v2_lag_profiles")
    display(frame[frame.statistic.isin(["median", "historical_median"])])

### V2 / T07 — Do several lags jointly show dependence?

The existing Ljung–Box implementation checks raw, absolute and squared returns at 10 and 20 lags. We report the proportion of individual paths with a nominal p-value below 0.05, separately for total returns and returns with prescribed jumps removed. Historical windows are shown too.

These are exploratory dependence diagnostics. A rejection proportion is not a model failure rate. A low p-value for absolute or squared returns can reflect the clustering we want to reproduce. The full Q-statistic, p-value summaries and histograms are exported.

In [ ]:
if v2_session is not None:
    frame = v2_session.table("v2_ljung_box")
    display(frame[frame.metric.eq("rejection_proportion")][
        ["return_series", "transform", "max_lag", "cohort", "value", "n_assessed", "n_excluded", "status"]])

### V2 / T08 — Does volatility persist across groups of days?

We reuse the existing 5-, 10- and 20-day block calculations. They measure the average size of returns, how much that size varies between blocks, and whether a volatile block tends to follow another volatile block. Incomplete trailing blocks are counted and excluded. A one-year path contains few long blocks, so their correlations are noisy descriptive estimates.

In [ ]:
if v2_session is not None:
    frame = v2_session.table("v2_volatility_blocks")
    display(frame[frame.statistic.eq("median")][
        ["return_series", "cohort", "block_days", "metric", "value", "n_assessed", "n_excluded",
         "complete_blocks_per_series", "discarded_returns_per_series", "status"]])

### Historical GARCH residuals — How much clustering remains after the fit?

We use the **actual pre-simulation calibration returns**, not simulated Gaussian draws. First subtract the fitted historical mean. Then divide that residual by the fitted conditional daily volatility. If GARCH explains the changing scale well, the standardized residuals should show less clustering.

ARCH-LM at lags 5, 10 and 20 regresses squared residuals on an intercept and their preceding lagged squares. We report `LM = (n − lag) × R²` against a chi-square distribution with `lag` degrees of freedom, with `ddof=0`, matching the documented statsmodels convention. The table includes the full and effective sample counts. The minimum reporting sample is `max(50, 5 × (lag + 1))`; missing, noncontiguous or degenerate samples are unassessed.

Dependence before standardization indicates historical clustering. Dependence afterward suggests that GARCH has not fully explained it. **A nonsignificant result does not prove the model is correct.** These nominal p-values assume an adequate conditional mean, a stationary sample, suitable finite moments and a large-sample approximation. The parameters were estimated on this same history, and no multiple-testing correction is applied, so these are exploratory diagnostics rather than automatic pass/fail tests.

Method: [statsmodels ARCH-LM documentation](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.het_arch.html). The comparison plot uses the existing T06 squared-return correlation convention.

In [ ]:
if v2_session is not None:
    v2_session.history()
    display(v2_session.table("garch_history_integrity"))
    display(v2_session.table("garch_arch_lm")[[
        "stage", "sample_start", "sample_end", "n_observations", "n_effective", "lag",
        "lm_statistic", "p_value", "status", "reason"]])
    fig = historical_residual_figure(v2_session)
    v2_session.figures["historical_squared_residual_acf"] = fig
    v2_show_figure(fig)

### Event volatility — Does the extra variance start and persist correctly?

We reconstruct the small GARCH recurrence using the recorded parameters and seed. **Before using a counterfactual, every reconstructed volatility cell and the random-draw digest must match the export.** We also verify the event-variance amounts against the recorded scenario settings.

Each event is tested separately: one simulation has no extra event variance, while its paired simulation has only that event's variance contribution. Both use identical initial states and random draws. All prescribed price jumps remain identical; price jumps never enter the GARCH innovations. The other variance contributions are disabled in both arms, so later shocks cannot obscure the event's decay. These checks use all latent volatility paths, including any paths whose tradable equity price is overridden to zero; latent volatility after zero is not tradable equity volatility.

We check no difference before the event, an increase of exactly `q` on the event day **before its random movement**, and the pathwise recurrence afterward. The expected average excess variance after `k` sessions is `q × (alpha + beta)^k`. Individual paths need not decline monotonically. The reported half-life, `log(0.5) / log(alpha + beta)`, is for **expected excess variance**, not volatility itself or the price level.

The charts show mean volatility and average excess variance, with pointwise Monte Carlo intervals. These intervals measure simulation sampling uncertainty, not historical realism or parameter uncertainty; occasional deviations are not automatic failures. Tables report 0, 1, 5, 20 and 60 sessions and flag truncated windows and overlapping scheduled events. Historical announcement data would still be needed to establish whether the assumed event effects are empirically realistic.

In [ ]:
if v2_session is not None:
    v2_session.events()
    display(v2_session.table("v2_replay_checks"))
    display(v2_session.table("v2_event_checks"))
    v2_show("v2_event_summary", [
        "event_id", "event_date", "event_variance_addition", "persistence_alpha_plus_beta",
        "expected_excess_variance_half_life_sessions", "available_post_event_sessions",
        "half_life_window_truncated", "pre_event_window_truncated", "window_60_sessions_truncated",
        "next_scheduled_variance_event_after_sessions", "previous_events_within_half_life",
        "other_scheduled_variance_event_within_half_life"])
    v2_show("v2_event_horizons", [
        "event_id", "requested_horizon_sessions", "mean_excess_variance",
        "theoretical_expected_excess_variance", "monte_carlo_standard_error",
        "standardized_mc_discrepancy", "remaining_effect_fraction_of_q", "window_truncated", "status"])
    for name, fig in event_figures(v2_session).items():
        v2_session.figures[name] = fig
        v2_show_figure(fig)

### V2 findings and exports

The findings below are calculated from this run, including weaknesses and unassessed results. All new tables are saved as JSON, Parquet and CSV, with source identity, hashes, method notes and execution status in a separate `dashboard_manifest.json`. Figures are saved as SVG and PNG. A readable HTML summary links to every result table. These outputs go to `V2_RESULTS_ROOT / company / source_run_id / validation_batch`; the earlier validation outputs are not replaced.

The new sections assess volatility diagnostics and scenario mechanics. They do not by themselves establish superior out-of-sample forecasts, empirical announcement realism, Gaussian tail adequacy, or portfolio performance.

In [ ]:
if v2_session is not None:
    v2_session._add("v2_findings", findings(v2_session))
    display(v2_session.table("v2_findings")[["test_id", "title", "interpretation"]])
    v2_result_folder = v2_session.export()
    write_report(v2_session, v2_result_folder)
    import json
    v2_manifest = json.loads((v2_result_folder / "dashboard_manifest.json").read_text())
    display(pd.DataFrame([{"table": d["name"], "rows": d["rows"], **d["files"]} for d in v2_manifest["datasets"]]))
    print(f"V2 results: {v2_result_folder}")
    print(f"Readable findings: {v2_result_folder / 'v2_validation_findings.html'}")